In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("OPENAI_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"OPENAI_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("OPENAI_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'OPENAI_API_KEY', and load_dotenv() ran without error.")

OPENAI_API_KEY loaded (164 characters): sk-p...VmEA


In [1]:
"""
Phase 2 triple extraction: full corpus, gpt-5.6-luna only.

Runs the Phase 2 extraction prompt against every chunk in chunks_all.parquet
(not a sample). Luna was selected after a 5-model comparison on cost and
signal-to-noise (see phase2_all_models_triples.xlsx and the model comparison
discussion): cheapest of the OpenAI tier, lowest citation-chain noise, and
GPT-5.5 Pro's ~150-1300x cost premium was not justified by the quality gain.

This is a long run (1000+ chunks), so unlike the 30-chunk pilot script this
version:
    - checkpoints progress to disk every CHECKPOINT_EVERY chunks, so a crash
      or interruption does not lose completed work
    - resumes automatically from the last checkpoint on re-run
    - retries transient connection/timeout errors with backoff before giving
      up on a chunk
    - writes the flattened triples table incrementally, not just at the end

Produces:
    - model_comparison_output/phase2_luna_full_raw.xlsx      one row per chunk
    - model_comparison_output/phase2_luna_full_triples.xlsx  one row per triple
    - model_comparison_output/phase2_luna_full_checkpoint.jsonl  resume state

Requires a .env file with:
    OPENAI_API_KEY=...

Usage:
    python run_phase2_luna_full.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "gpt-5.6-luna"
MODEL_PRICING = {"gpt-5.6-luna": (1.00, 6.00)}  # $/1M tokens, (input, output)

CHECKPOINT_PATH = OUTPUT_DIR / "phase2_luna_full_checkpoint.jsonl"
RAW_OUTPUT_PATH = OUTPUT_DIR / "phase2_luna_full_raw.xlsx"
TRIPLES_OUTPUT_PATH = OUTPUT_DIR / "phase2_luna_full_triples.xlsx"

CHECKPOINT_EVERY = 25     # flush checkpoint to disk every N chunks

# Connection-type errors (wifi drop, DNS blip, timeout) are treated as
# recoverable and retried patiently, since the run is long and unattended.
# Non-connection errors (e.g. a genuine 400 on malformed input) are not
# worth retrying the same way, they fail fast after a couple of tries.
CONNECTION_MAX_RETRIES = 20      # up to 20 retries for connection-type errors
CONNECTION_RETRY_BACKOFF_SEC = 15  # 15,30,60,120,240,300(capped)...
CONNECTION_RETRY_CAP_SEC = 300     # never wait longer than 5 min between retries
OTHER_MAX_RETRIES = 3
OTHER_RETRY_BACKOFF_SEC = 10


# ---------------------------------------------------------------------------
# Fail fast on a bad key
# ---------------------------------------------------------------------------

def verify_openai_key():
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    try:
        client.models.list()
    except Exception as e:
        raise RuntimeError(
            f"OpenAI API key check failed before running any chunks: {e}\n"
            "Fix OPENAI_API_KEY in .env and re-run."
        )
    print("OpenAI API key verified.")


# ---------------------------------------------------------------------------
# Load prompt + full corpus
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def load_full_corpus() -> pd.DataFrame:
    df = pd.read_parquet(CHUNKS_PATH)
    before = len(df)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    print(f"Loaded {before} chunks, {len(df)} after length filter (>200 chars).")
    return df


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
corpus_df = load_full_corpus()


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"


# ---------------------------------------------------------------------------
# Checkpointing: one JSON line per completed chunk result
# ---------------------------------------------------------------------------

def load_checkpoint() -> dict:
    """Returns {chunk_id: result_dict} for chunks completed successfully.

    Chunks that permanently failed (parse_error set after retries exhausted)
    are NOT treated as done here, so a fresh run automatically retries them
    again rather than leaving them lost forever, the way the old 30-chunk
    comparison script left Pro's 10 connection-error chunks unrecovered.
    Successful chunks (parse_error is None) are the only ones skipped.
    """
    done = {}
    failed_count = 0
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    if rec.get("parse_error") is None:
                        done[rec["chunk_id"]] = rec
                    else:
                        failed_count += 1
                except json.JSONDecodeError:
                    continue
        msg = f"Resuming: {len(done)} chunks completed successfully in checkpoint."
        if failed_count:
            msg += f" {failed_count} previously-failed chunk(s) will be retried."
        print(msg)
    return done


def load_all_checkpoint_records() -> dict:
    """Returns {chunk_id: latest_record} for every chunk ever attempted,
    successful or not. Used only for final reporting, so permanently-failed
    chunks remain visible in the output files rather than disappearing.
    If a chunk_id appears more than once (retried across separate runs),
    the most recent (last) record for that id wins.
    """
    records = {}
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    records[rec["chunk_id"]] = rec
                except json.JSONDecodeError:
                    continue
    return records


def append_checkpoint(record: dict):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


# ---------------------------------------------------------------------------
# Luna caller with retry on transient errors
# ---------------------------------------------------------------------------

def call_luna(client, user_msg: str):
    """gpt-5.6-luna rejects an explicit temperature other than the default,
    so temperature is omitted rather than set to 0.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )
    raw_output = response.choices[0].message.content
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else None
    output_tokens = usage.completion_tokens if usage else None
    return raw_output, input_tokens, output_tokens


def run_chunk_with_retry(client, row: pd.Series) -> dict:
    user_msg = build_user_message(row)
    attempt = 0
    while True:
        attempt += 1
        start = time.time()
        try:
            raw_output, input_tokens, output_tokens = call_luna(client, user_msg)
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)
            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            }
        except Exception as e:
            err_str = str(e)
            is_connection = any(s in err_str for s in
                                 ["Connection", "connection", "timeout", "Timeout",
                                  "503", "502", "504", "429", "APIConnectionError",
                                  "RemoteProtocolError", "ReadTimeout"])

            if is_connection:
                max_retries = CONNECTION_MAX_RETRIES
                wait = min(
                    CONNECTION_RETRY_BACKOFF_SEC * (2 ** (attempt - 1)),
                    CONNECTION_RETRY_CAP_SEC,
                )
            else:
                max_retries = OTHER_MAX_RETRIES
                wait = OTHER_RETRY_BACKOFF_SEC * (2 ** (attempt - 1))

            if attempt <= max_retries:
                kind = "connection" if is_connection else "other"
                print(f"    retry {attempt}/{max_retries} ({kind} error) for "
                      f"{row['id']} in {wait}s ({err_str[:120]})")
                time.sleep(wait)
                continue

            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error after {attempt} attempt(s): {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            }


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    flat_rows = []
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            flat_row = {
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    from openai import OpenAI

    verify_openai_key()
    client = OpenAI(api_key=OPENAI_API_KEY)

    done = load_checkpoint()
    remaining = corpus_df[~corpus_df["id"].isin(done.keys())].reset_index(drop=True)
    print(f"{len(remaining)} chunks remaining out of {len(corpus_df)} total.\n")

    since_last_flush = 0
    start_time = time.time()

    for i, row in remaining.iterrows():
        result = run_chunk_with_retry(client, row)
        append_checkpoint(result)
        since_last_flush += 1

        status = "ok" if result["parse_error"] is None else "FAILED"
        n_tri = result["n_triples"] if result["n_triples"] is not None else "-"
        lat = f"{result['latency_sec']:.1f}s" if result["latency_sec"] else "-"
        overall_done = len(done) + i + 1
        print(f"  [{overall_done}/{len(corpus_df)}] {status} "
              f"({n_tri} triples, {lat}) {row['id']}")

        if since_last_flush >= CHECKPOINT_EVERY:
            elapsed_min = (time.time() - start_time) / 60
            print(f"  --- checkpoint: {overall_done}/{len(corpus_df)} done, "
                  f"{elapsed_min:.1f} min elapsed ---")
            since_last_flush = 0

    print("\nAll chunks processed. Building final output files...")

    # Reload everything from checkpoint (source of truth) rather than
    # relying on in-memory state, in case this run resumed a prior one.
    # Uses load_all_checkpoint_records so permanently-failed chunks still
    # show up in the raw output file for visibility, not just successes.
    all_records = load_all_checkpoint_records()
    combined = pd.DataFrame(list(all_records.values()))
    combined["model"] = MODEL_NAME
    combined = add_cost_column(combined)

    combined.to_excel(RAW_OUTPUT_PATH, index=False)
    print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")

    triples_df = flatten_triples(combined)
    triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
    print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")

    n_failed = combined["parse_error"].notna().sum()
    total_cost = combined["est_cost_usd"].sum()
    total_latency_hr = combined["latency_sec"].sum() / 3600 if combined["latency_sec"].notna().any() else 0
    print(f"\nChunks run: {len(combined)}  |  Failed: {n_failed}  |  "
          f"Est. total cost: ${total_cost:.2f}  |  "
          f"Total model time: {total_latency_hr:.1f} hr")
    if n_failed:
        print(f"\n{n_failed} chunks failed after retries. Re-running this script "
              f"will retry only the failed/missing chunks (resume is automatic).")


if __name__ == "__main__":
    main()

Loaded 1108 chunks, 1104 after length filter (>200 chars).
OpenAI API key verified.
1104 chunks remaining out of 1104 total.

  [1/1104] ok (21 triples, 29.6s) 10-1002_cche-10383_abstract1
  [2/1104] ok (4 triples, 9.9s) 10-1002_cche-10383_intro1
  [3/1104] ok (3 triples, 9.4s) 10-1002_cche-10383_methods1
  [4/1104] ok (18 triples, 28.9s) 10-1002_cche-10383_results_discussion1
  [5/1104] ok (25 triples, 39.7s) 10-1002_cche-10383_results_discussion2
  [6/1104] ok (17 triples, 27.0s) 10-1002_cche-10589_abstract1
  [7/1104] ok (0 triples, 4.1s) 10-1002_cche-10589_intro1
  [8/1104] ok (0 triples, 4.2s) 10-1002_cche-10589_methods1
  [9/1104] ok (0 triples, 4.6s) 10-1002_cche-10589_methods2
  [10/1104] ok (26 triples, 44.6s) 10-1002_cche-10589_results_discussion1
  [11/1104] ok (18 triples, 32.6s) 10-1002_cche-10589_results_discussion2
  [12/1104] ok (15 triples, 32.5s) 10-1002_cche-10589_results_discussion3
  [13/1104] ok (18 triples, 27.9s) 10-1002_cche-10589_conclusion1
  [14/1104] ok (0 

IllegalCharacterError: [
  {
    "pmid": "10-1007_s13197-019-03785-8",
    "source": "alkaline extraction",
    "source_type": "extraction_method",
    "material": "sour cherry kernel flour",
    "interaction": "decreased",
    "target": "lipid content",
    "target_type": "physicochemical_property",
    "compared_property": "",
    "reported_value": "from 34.75 ± 0.68% to 9.00 ± 0.28%",
    "claim_status": "observed",
    "flagged_phrase": "",
    "corresponding_sentence": "After extraction, the lipid content of the kernel ﬂour was reduced from 34.75 ± 0.68% to 9.00 ± 0.28%, indicating lower lipid extraction efﬁciency."
  },
  {
    "pmid": "10-1007_s13197-019-03785-8",
    "source": "sour cherry kernel protein concentrate",
    "source_type": "material_used_directly",
    "material": "",
    "interaction": "exhibited",
    "target": "apparent molecular weight distribution",
    "target_type": "other",
    "compared_property": "",
    "reported_value": "14–66 kDa",
    "claim_status":

# Rebuild

In [2]:
"""
Phase 2 triple extraction: full corpus, gpt-5.6-luna only.

Runs the Phase 2 extraction prompt against every chunk in chunks_all.parquet
(not a sample). Luna was selected after a 5-model comparison on cost and
signal-to-noise (see phase2_all_models_triples.xlsx and the model comparison
discussion): cheapest of the OpenAI tier, lowest citation-chain noise, and
GPT-5.5 Pro's ~150-1300x cost premium was not justified by the quality gain.

This is a long run (1000+ chunks), so unlike the 30-chunk pilot script this
version:
    - checkpoints progress to disk every CHECKPOINT_EVERY chunks, so a crash
      or interruption does not lose completed work
    - resumes automatically from the last checkpoint on re-run
    - retries transient connection/timeout errors with backoff before giving
      up on a chunk
    - writes the flattened triples table incrementally, not just at the end

Produces:
    - model_comparison_output/phase2_luna_full_raw.xlsx      one row per chunk
    - model_comparison_output/phase2_luna_full_triples.xlsx  one row per triple
    - model_comparison_output/phase2_luna_full_checkpoint.jsonl  resume state

Requires a .env file with:
    OPENAI_API_KEY=...

Usage:
    python run_phase2_luna_full.py
"""

import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY not found. Add it to your .env file.")

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

CHUNKS_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PDF_PREPROCESSING_INTO_CHUNKS\chunks_all.parquet"
PROMPT_PATH = "phase2_extraction_prompt.md"
OUTPUT_DIR = Path("model_comparison_output")
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "gpt-5.6-luna"
MODEL_PRICING = {"gpt-5.6-luna": (1.00, 6.00)}  # $/1M tokens, (input, output)

CHECKPOINT_PATH = OUTPUT_DIR / "phase2_luna_full_checkpoint.jsonl"
RAW_OUTPUT_PATH = OUTPUT_DIR / "phase2_luna_full_raw.xlsx"
TRIPLES_OUTPUT_PATH = OUTPUT_DIR / "phase2_luna_full_triples.xlsx"

CHECKPOINT_EVERY = 25     # flush checkpoint to disk every N chunks

# Connection-type errors (wifi drop, DNS blip, timeout) are treated as
# recoverable and retried patiently, since the run is long and unattended.
# Non-connection errors (e.g. a genuine 400 on malformed input) are not
# worth retrying the same way, they fail fast after a couple of tries.
CONNECTION_MAX_RETRIES = 20      # up to 20 retries for connection-type errors
CONNECTION_RETRY_BACKOFF_SEC = 15  # 15,30,60,120,240,300(capped)...
CONNECTION_RETRY_CAP_SEC = 300     # never wait longer than 5 min between retries
OTHER_MAX_RETRIES = 3
OTHER_RETRY_BACKOFF_SEC = 10


# ---------------------------------------------------------------------------
# Fail fast on a bad key
# ---------------------------------------------------------------------------

def verify_openai_key():
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    try:
        client.models.list()
    except Exception as e:
        raise RuntimeError(
            f"OpenAI API key check failed before running any chunks: {e}\n"
            "Fix OPENAI_API_KEY in .env and re-run."
        )
    print("OpenAI API key verified.")


# ---------------------------------------------------------------------------
# Load prompt + full corpus
# ---------------------------------------------------------------------------

def load_system_prompt(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"^#\s+Phase 2.*\n", "", text, count=1)
    return text.strip()


def load_full_corpus() -> pd.DataFrame:
    df = pd.read_parquet(CHUNKS_PATH)
    before = len(df)
    df = df[df["chunk_text"].str.strip().str.len() > 200].reset_index(drop=True)
    print(f"Loaded {before} chunks, {len(df)} after length filter (>200 chars).")
    return df


SYSTEM_PROMPT = load_system_prompt(PROMPT_PATH)
corpus_df = load_full_corpus()


def build_user_message(row: pd.Series) -> str:
    return (
        f"pmid: {row['doi']}\n"
        f"section: {row['section']}\n\n"
        f"chunk_text:\n{row['chunk_text']}"
    )


# ---------------------------------------------------------------------------
# JSON parsing helper
# ---------------------------------------------------------------------------

def parse_triples(raw_text: str):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed, None
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"


# ---------------------------------------------------------------------------
# Checkpointing: one JSON line per completed chunk result
# ---------------------------------------------------------------------------

def load_checkpoint() -> dict:
    """Returns {chunk_id: result_dict} for chunks completed successfully.

    Chunks that permanently failed (parse_error set after retries exhausted)
    are NOT treated as done here, so a fresh run automatically retries them
    again rather than leaving them lost forever, the way the old 30-chunk
    comparison script left Pro's 10 connection-error chunks unrecovered.
    Successful chunks (parse_error is None) are the only ones skipped.
    """
    done = {}
    failed_count = 0
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    if rec.get("parse_error") is None:
                        done[rec["chunk_id"]] = rec
                    else:
                        failed_count += 1
                except json.JSONDecodeError:
                    continue
        msg = f"Resuming: {len(done)} chunks completed successfully in checkpoint."
        if failed_count:
            msg += f" {failed_count} previously-failed chunk(s) will be retried."
        print(msg)
    return done


def load_all_checkpoint_records() -> dict:
    """Returns {chunk_id: latest_record} for every chunk ever attempted,
    successful or not. Used only for final reporting, so permanently-failed
    chunks remain visible in the output files rather than disappearing.
    If a chunk_id appears more than once (retried across separate runs),
    the most recent (last) record for that id wins.
    """
    records = {}
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                    records[rec["chunk_id"]] = rec
                except json.JSONDecodeError:
                    continue
    return records


def append_checkpoint(record: dict):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


# ---------------------------------------------------------------------------
# Luna caller with retry on transient errors
# ---------------------------------------------------------------------------

def call_luna(client, user_msg: str):
    """gpt-5.6-luna rejects an explicit temperature other than the default,
    so temperature is omitted rather than set to 0.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )
    raw_output = response.choices[0].message.content
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else None
    output_tokens = usage.completion_tokens if usage else None
    return raw_output, input_tokens, output_tokens


def run_chunk_with_retry(client, row: pd.Series) -> dict:
    user_msg = build_user_message(row)
    attempt = 0
    while True:
        attempt += 1
        start = time.time()
        try:
            raw_output, input_tokens, output_tokens = call_luna(client, user_msg)
            elapsed = time.time() - start
            triples, err = parse_triples(raw_output)
            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": raw_output,
                "n_triples": len(triples) if triples is not None else None,
                "parse_error": err,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "latency_sec": round(elapsed, 2),
            }
        except Exception as e:
            err_str = str(e)
            is_connection = any(s in err_str for s in
                                 ["Connection", "connection", "timeout", "Timeout",
                                  "503", "502", "504", "429", "APIConnectionError",
                                  "RemoteProtocolError", "ReadTimeout"])

            if is_connection:
                max_retries = CONNECTION_MAX_RETRIES
                wait = min(
                    CONNECTION_RETRY_BACKOFF_SEC * (2 ** (attempt - 1)),
                    CONNECTION_RETRY_CAP_SEC,
                )
            else:
                max_retries = OTHER_MAX_RETRIES
                wait = OTHER_RETRY_BACKOFF_SEC * (2 ** (attempt - 1))

            if attempt <= max_retries:
                kind = "connection" if is_connection else "other"
                print(f"    retry {attempt}/{max_retries} ({kind} error) for "
                      f"{row['id']} in {wait}s ({err_str[:120]})")
                time.sleep(wait)
                continue

            return {
                "chunk_id": row["id"],
                "doi": row["doi"],
                "section": row["section"],
                "raw_output": None,
                "n_triples": None,
                "parse_error": f"API error after {attempt} attempt(s): {e}",
                "input_tokens": None,
                "output_tokens": None,
                "latency_sec": None,
            }


# ---------------------------------------------------------------------------
# Cost estimate
# ---------------------------------------------------------------------------

def add_cost_column(df: pd.DataFrame) -> pd.DataFrame:
    in_price, out_price = MODEL_PRICING[MODEL_NAME]
    def _cost(r):
        if pd.isna(r["input_tokens"]) or pd.isna(r["output_tokens"]):
            return None
        return (r["input_tokens"] / 1_000_000 * in_price) + (r["output_tokens"] / 1_000_000 * out_price)
    df["est_cost_usd"] = df.apply(_cost, axis=1)
    return df


# ---------------------------------------------------------------------------
# Flatten raw_output JSON into one row per triple
# ---------------------------------------------------------------------------

TRIPLE_FIELDS = [
    "pmid", "source", "source_type", "material", "interaction", "target",
    "target_type", "compared_property", "reported_value", "claim_status",
    "flagged_phrase", "corresponding_sentence",
]


def flatten_triples(raw_df: pd.DataFrame) -> pd.DataFrame:
    flat_rows = []
    for _, row in raw_df.iterrows():
        if pd.isna(row["raw_output"]):
            continue
        triples, err = parse_triples(row["raw_output"])
        if triples is None:
            continue
        for t in triples:
            flat_row = {
                "chunk_id": row["chunk_id"],
                "doi": row["doi"],
                "section": row["section"],
            }
            for field in TRIPLE_FIELDS:
                flat_row[field] = t.get(field, "")
            flat_rows.append(flat_row)
    return pd.DataFrame(flat_rows)


# ---------------------------------------------------------------------------
# Excel-safe string sanitization
# ---------------------------------------------------------------------------

# openpyxl (used by pandas.to_excel) rejects certain control characters that
# XML cannot encode. These can show up in raw_output (rare model artifacts)
# or, more commonly, inside parse_error messages that embed the entire raw
# response text when JSON parsing fails. Without sanitizing, a single bad
# cell can throw IllegalCharacterError and abort the whole file write, even
# after every chunk has already been successfully processed and checkpointed.
_ILLEGAL_XLSX_CHARS_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"  # control chars except \t \n \r
)


def sanitize_for_excel(value):
    if isinstance(value, str):
        return _ILLEGAL_XLSX_CHARS_RE.sub("", value)
    return value


def sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    try:
        return df.map(sanitize_for_excel)          # pandas >= 2.1
    except AttributeError:
        return df.applymap(sanitize_for_excel)      # pandas < 2.1


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    from openai import OpenAI

    verify_openai_key()
    client = OpenAI(api_key=OPENAI_API_KEY)

    done = load_checkpoint()
    remaining = corpus_df[~corpus_df["id"].isin(done.keys())].reset_index(drop=True)
    print(f"{len(remaining)} chunks remaining out of {len(corpus_df)} total.\n")

    since_last_flush = 0
    start_time = time.time()

    for i, row in remaining.iterrows():
        result = run_chunk_with_retry(client, row)
        append_checkpoint(result)
        since_last_flush += 1

        status = "ok" if result["parse_error"] is None else "FAILED"
        n_tri = result["n_triples"] if result["n_triples"] is not None else "-"
        lat = f"{result['latency_sec']:.1f}s" if result["latency_sec"] else "-"
        overall_done = len(done) + i + 1
        print(f"  [{overall_done}/{len(corpus_df)}] {status} "
              f"({n_tri} triples, {lat}) {row['id']}")

        if since_last_flush >= CHECKPOINT_EVERY:
            elapsed_min = (time.time() - start_time) / 60
            print(f"  --- checkpoint: {overall_done}/{len(corpus_df)} done, "
                  f"{elapsed_min:.1f} min elapsed ---")
            since_last_flush = 0

    print("\nAll chunks processed. Building final output files...")

    # Reload everything from checkpoint (source of truth) rather than
    # relying on in-memory state, in case this run resumed a prior one.
    # Uses load_all_checkpoint_records so permanently-failed chunks still
    # show up in the raw output file for visibility, not just successes.
    all_records = load_all_checkpoint_records()
    combined = pd.DataFrame(list(all_records.values()))
    combined["model"] = MODEL_NAME
    combined = add_cost_column(combined)
    combined = sanitize_dataframe_for_excel(combined)

    try:
        combined.to_excel(RAW_OUTPUT_PATH, index=False)
        print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")
    except Exception as e:
        # Never lose a completed run over a write-format issue. Fall back to
        # CSV, which has no character restrictions, so the data survives
        # even if something Excel-specific still trips up openpyxl.
        csv_fallback = RAW_OUTPUT_PATH.with_suffix(".csv")
        combined.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    triples_df = flatten_triples(combined)
    triples_df = sanitize_dataframe_for_excel(triples_df)
    try:
        triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
        print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")
    except Exception as e:
        csv_fallback = TRIPLES_OUTPUT_PATH.with_suffix(".csv")
        triples_df.to_csv(csv_fallback, index=False, encoding="utf-8")
        print(f"WARNING: Excel write failed ({e}). "
              f"Saved as CSV instead -> {csv_fallback}")

    n_failed = combined["parse_error"].notna().sum()
    total_cost = combined["est_cost_usd"].sum()
    total_latency_hr = combined["latency_sec"].sum() / 3600 if combined["latency_sec"].notna().any() else 0
    print(f"\nChunks run: {len(combined)}  |  Failed: {n_failed}  |  "
          f"Est. total cost: ${total_cost:.2f}  |  "
          f"Total model time: {total_latency_hr:.1f} hr")
    if n_failed:
        print(f"\n{n_failed} chunks failed after retries. Re-running this script "
              f"will retry only the failed/missing chunks (resume is automatic).")


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "--rebuild-only":
        # Regenerate output files from the existing checkpoint without
        # calling the API again. Use this if extraction already finished
        # (or got far enough) but the final Excel write failed.
        print("Rebuilding output files from checkpoint only, no API calls...")
        all_records = load_all_checkpoint_records()
        combined = pd.DataFrame(list(all_records.values()))
        combined["model"] = MODEL_NAME
        combined = add_cost_column(combined)
        combined = sanitize_dataframe_for_excel(combined)

        try:
            combined.to_excel(RAW_OUTPUT_PATH, index=False)
            print(f"Raw results ({len(combined)} chunks) -> {RAW_OUTPUT_PATH}")
        except Exception as e:
            csv_fallback = RAW_OUTPUT_PATH.with_suffix(".csv")
            combined.to_csv(csv_fallback, index=False, encoding="utf-8")
            print(f"WARNING: Excel write failed ({e}). Saved as CSV -> {csv_fallback}")

        triples_df = flatten_triples(combined)
        triples_df = sanitize_dataframe_for_excel(triples_df)
        try:
            triples_df.to_excel(TRIPLES_OUTPUT_PATH, index=False)
            print(f"Flattened triples ({len(triples_df)} rows) -> {TRIPLES_OUTPUT_PATH}")
        except Exception as e:
            csv_fallback = TRIPLES_OUTPUT_PATH.with_suffix(".csv")
            triples_df.to_csv(csv_fallback, index=False, encoding="utf-8")
            print(f"WARNING: Excel write failed ({e}). Saved as CSV -> {csv_fallback}")

        n_failed = combined["parse_error"].notna().sum()
        print(f"\nChunks in checkpoint: {len(combined)}  |  Failed: {n_failed}")
    else:
        main()

Loaded 1108 chunks, 1104 after length filter (>200 chars).
OpenAI API key verified.
Resuming: 1090 chunks completed successfully in checkpoint. 14 previously-failed chunk(s) will be retried.
14 chunks remaining out of 1104 total.

  [1091/1104] ok (12 triples, 26.9s) 10-1007_s13197-019-03785-8_results_discussion1
  [1092/1104] ok (8 triples, 16.1s) 10-1111_ijfs-13271_results1
  [1093/1104] FAILED (- triples, 9.1s) 10-1111_ijfs-15446_methods1
  [1094/1104] FAILED (- triples, 25.4s) 10-1111_ijfs-15446_results_discussion1
  [1095/1104] ok (21 triples, 29.3s) 10-1111_ijfs-15831_results_discussion1
  [1096/1104] FAILED (- triples, 29.9s) 10-1111_ijfs-16923_results_discussion1
  [1097/1104] FAILED (- triples, 24.6s) 10-1111_ijfs-17244_results_discussion2
  [1098/1104] FAILED (- triples, 27.4s) 10-1111_ijfs-17596_results_discussion2
  [1099/1104] FAILED (- triples, 17.2s) 10-1111_jfpe-14243_results_discussion2
  [1100/1104] FAILED (- triples, 20.7s) 10-1111_jfpe-14578_results_discussion1
  [1